# M01 — Gravity station corrections

**Level 2 · Prerequisites: foundations notebook; read time: 35 minutes.**

## Objectives

This lesson executes the product station processor on its committed analytic control. It targets Bouguer disturbance, not density inversion. Gravity is downward-positive mGal; heights are upward WGS84 ellipsoidal metres. The authored control is not field data.

## Setup

Use the already reviewed Python 3.12 pipeline environment. This notebook installs nothing.

Import repository code and confirm its source file.

In [ ]:
import json, sys
from pathlib import Path
root=Path.cwd().parents[2]; sys.path.insert(0,str(root/'data-pipeline'))
from gravity_processing import process_survey
# Identify the implementation loaded from this repository.
print(Path(__import__('gravity_processing').__file__).resolve())

## Data

This lesson executes the product station processor on its committed analytic control. It targets Bouguer disturbance, not density inversion. Gravity is downward-positive mGal; heights are upward WGS84 ellipsoidal metres. The authored control is not field data.

**Files:** docs/methods/gravity-processing/examples/station-control.json

## Method

For latitude φ and receiver height h, D=g−γ(φ,h). The infinite land-plate attraction is B=2πGρt×10⁵ mGal, where G is m³ kg⁻¹ s⁻², ρ is kg m⁻³ and surface thickness t is m. Output D_B=D−B. Receiver height and terrain thickness are distinct; this is not automatic free-air or DEM terrain correction.

Execute the product method on the declared control.

In [ ]:
request=json.loads((root/'docs/methods/gravity-processing/examples/station-control.json').read_text(encoding='utf-8'))
# Execute the product correction core on its committed analytic control.
result=process_survey(request['dataset'],request['config'])
print({'state':result['dataset']['state'],'derived_mGal':result['qc']['derived_mgal'],'uncertainty_mGal':result['processing']['uncertainty_mgal'],'accepted':result['processing']['full_method_accepted']})

Adversarial or negative control: isolate one assumption or violate one declared physical constraint.

In [ ]:
try:
    process_survey(result['dataset'],request['config'])
except ValueError as error:
    # Reapplying an already reached correction state must be refused.
    print('Expected duplicate-correction refusal:',str(error))
else:
    raise AssertionError('Repeated correction was accepted')

USGS describes upstream acquisition and QA. Boule calculates height-aware normal gravity; Harmonica implements plate attraction. Conditional uncertainty propagates declared marginal errors; it is not geological confidence. One analytic station cannot evaluate regional maps, terrain or field eligibility.

## Results

Expect about 12 mGal, an ordered correction history and `full_method_accepted=false`. Reapplication must fail. This is local arithmetic/state evidence, not full M01 acceptance.

## Summary

A correction is a typed ordered transformation. Preserve units, reference, original value and history; never infer missing metadata from numeric magnitude.



## References

- https://pubs.usgs.gov/tm/02/d04/tm2d4.pdf
- https://www.fatiando.org/harmonica/latest/api/index.html